# 3．KD alpha 对比：12 次新增训练，仅比较验证集

本 notebook 在**独立的新实验目录**中比较两种 KD 的 alpha，不改动 v1，也不运行任何测试集评估。

|项目|设置|
|---|---|
|方法|原始 KD、校准 KD|
|alpha|0.25、0.5、0.75|
|alpha 定义|KD 项权重：`L = (1-alpha) × CE + alpha × tau² × KL(Teacher || Student)`|
|tau|固定 2；与 Teacher 校准温度 T 不同|
|种子|20261010、20261011、20261012，所有设置共用|
|训练预算|30 epochs；batch size 128；AdamW，lr=0.001、weight_decay=0.0001|
|网络|冻结 CLIP 的 2048 维配对特征 → Linear(512) → ReLU → Dropout(0.1) → Linear(3)|
|最佳 epoch|val_select Macro-F1 最大；平局取 NLL 小者；再平局取较早 epoch|
|复用|alpha=0.5 的六个已选模型和完整训练历史|
|新增|2 种方法 × 2 个新 alpha × 3 个种子 = **12 次**|

交付：验证集均值±样本标准差表、逐种子最佳 epoch 和分数、完成项与恢复点、下一轮候选 alpha。三个种子全部保留，不挑最好种子。

**全部 18 组（含复用 6 组）完成后才给出候选。**不完整设置不计算部分种子的均值；即使某个方法先跑完，也不提前宣布候选。验证集已用于选择 epoch 和 alpha，候选仅用于下一轮计划，不是独立测试或显著性结论。


## 1．环境与输入

建议回到跑 v1 的同一 Colab GPU 环境。本 notebook 在训练前核对原记录的 PyTorch、NumPy、CUDA 版本；恢复时还核对 GPU 型号、代码和配置，避免将环境变化混进 alpha 比较。原包记录为 torch `2.11.0+cu130`、NumPy `2.2.6`、CUDA `13.0`。

需要 NumPy、pandas、PyTorch 和 packaging。不会自动安装或升级环境。汇总与输入检查不要求 GPU；十二次新训练要求原 CUDA 环境。

Windows 默认使用你提供的 ZIP；Colab 默认使用 Drive 中的同名 ZIP，请根据实际上传位置修改。也可以直接把 SOURCE 改为 v1 解压目录，以直接读取现有特征。ZIP 模式仅提取训练/验证所需文件，不提取测试集。


In [1]:
! pip install numpy==2.2.6

In [2]:
import sys
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    SOURCE = Path('/content/drive/MyDrive/CLIP_Jev_Student_v1')
    EXPERIMENT_DIR = Path('/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1')
else:
    SOURCE = Path('C:/Users/xhyha/xwechat_files/wxid_raynutuxt57v22_afa2/msg/file/2026-10/student_delivery.zip')
    EXPERIMENT_DIR = Path.cwd() / 'CLIP_Jev_alpha_sweep_v1'

# 首次运行新建此目录；断线后使用相同路径继续，不要每天新建目录。
# 若改变实验协议，请换一个新目录名，保留旧实验。
print('只读来源：', SOURCE)
print('新实验/恢复目录：', EXPERIMENT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
只读来源： /content/drive/MyDrive/CLIP_Jev_Student_v1
新实验/恢复目录： /content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1


## 2．加载训练与恢复函数

本格内嵌完整实现，可独立运行，不依赖旁边的 `.py` 文件。算法沿用 v1，只有两个新增 alpha 的 CE/KD 权重变化。新模型、历史、恢复状态只写入新目录；原 alpha=0.5 结果按哈希验证后复用。

checkpoint 使用独立临时文件原子保存。恢复点包含模型、AdamW 状态、随机数状态、最佳模型及历史；每 5 个完整 epoch 保存一次，与 v1 保持一致。


In [3]:
RUNTIME_SOURCE = r'''"""Validation-only alpha sweep. Embedded verbatim in the delivered notebook."""
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone, timedelta
import hashlib
import json
import os
import random
import shutil
import time
import uuid
import zipfile
import numpy as np
import pandas as pd

METHODS = ['KD_raw', 'KD_calibrated']
SEEDS = [20261010, 20261011, 20261012]
ALPHAS = [.25, .5, .75]


def check(value, message):
    if not bool(value):
        raise ValueError(message)


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def read_rows(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def atomic_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.partial')
    with temp.open('x', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
        f.flush(); os.fsync(f.fileno())
    temp.replace(path)


def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def score(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    cm = np.zeros((3, 3), dtype=int)
    np.add.at(cm, (y, p.argmax(1)), 1)
    f1 = 2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1), 1)
    return dict(macro_f1=float(f1.mean()), accuracy=float((p.argmax(1) == y).mean()),
                nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()),
                N_recall=float(cm[1, 1]/max(cm[1].sum(), 1)))


def run_name(method, alpha, seed):
    return f'{method}_alpha{alpha:.2f}_{seed}'


def prepare(source, experiment):
    """Read only source train/val files; create or resume a dedicated sweep directory."""
    source, experiment = Path(source), Path(experiment)
    check(source.exists(), 'SOURCE 不存在，请修改路径。')
    if source.is_dir():
        check(not experiment.resolve().is_relative_to(source.resolve()), '新实验目录不能放在旧实验内部。')
    marker = experiment / 'EXPERIMENT.json'
    if experiment.exists():
        check(marker.is_file(), '目录已经存在但不是本 notebook 创建的实验，请改用新目录。')
        check(read_json(marker) == {'kind': 'alpha_sweep_v1', 'source': str(source.resolve())}, '目录属于另一个实验或输入来源。')
    else:
        experiment.mkdir(parents=True, exist_ok=False)
        atomic_json(marker, {'kind': 'alpha_sweep_v1', 'source': str(source.resolve())})
    required = ['students/selection.json', 'students/training_lock.json', 'students/history.jsonl',
                'features/clip_revision.json', 'handoff/results/calibration.json',
                'handoff/predictions/train/jev.jsonl']
    for split in ['train', 'val_select']:
        required += [f'features/{split}.npz', f'features/{split}_meta.json', f'handoff/manifests/{split}.jsonl']
    if source.is_file():
        work = experiment / 'reused_inputs'
        with zipfile.ZipFile(source) as z:
            check(len(z.namelist()) == len(set(z.namelist())), 'ZIP 存在重复路径。')
            selection = json.loads(z.read('students/selection.json'))
            names = [c['checkpoint'] for c in selection['configs'] if c['method'] in METHODS]
            check(all(PurePosixPath(n).name == n and chr(92) not in n and ':' not in n for n in names), 'checkpoint 路径不合法。')
            required += ['students/' + n for n in names]
            for name in required:
                dest = work / name
                # Read ZIP contents to detect a changed source, without extracting any test files.
                h = hashlib.sha256()
                with z.open(name) as f:
                    for block in iter(lambda: f.read(1024*1024), b''):
                        h.update(block)
                if dest.exists():
                    check(digest(dest) == h.hexdigest(), f'复用文件已改变：{name}')
                    continue
                dest.parent.mkdir(parents=True, exist_ok=True)
                tmp = dest.with_name(dest.name + '.' + uuid.uuid4().hex + '.partial')
                with z.open(name) as src, tmp.open('xb') as dst:
                    shutil.copyfileobj(src, dst)
                check(digest(tmp) == h.hexdigest(), f'解压校验失败：{name}')
                tmp.replace(dest)
    else:
        work = source
        selection = read_json(work / 'students/selection.json')
        names = [c['checkpoint'] for c in selection['configs'] if c['method'] in METHODS]
        check(all(PurePosixPath(n).name == n and chr(92) not in n and ':' not in n for n in names), 'checkpoint 路径不合法。')
        required += ['students/' + n for n in names]
    check(all((work / n).is_file() for n in required), '缺少训练、验证或基线文件。')
    lock = read_json(work / 'students/training_lock.json')
    expected_settings = dict(epochs=30, seeds=SEEDS, alpha=.5, tau=2., batch_size=128,
                             learning_rate=.001, weight_decay=.0001, checkpoint_every_epochs=5)
    for key, value in expected_settings.items():
        check(lock[key] == value, f'已有训练设置与预期不同：{key}')
    check(selection['alpha'] == .5 and selection['tau'] == 2. and selection['epochs'] == 30, '基线配置不匹配。')
    configs = [c for c in selection['configs'] if c['method'] in METHODS]
    check(len(configs) == 6 and {(c['method'], c['seed']) for c in configs} == {(m,s) for m in METHODS for s in SEEDS}, 'alpha=.5 必须有六组完整结果。')
    check(len(set(names)) == 6, '基线 checkpoint 重复。')
    for c in configs:
        check(digest(work / 'students' / c['checkpoint']) == c['checkpoint_sha256'], '基线 checkpoint 哈希错误。')
    history = read_rows(work / 'students/history.jsonl')
    baseline = []
    for c in configs:
        h = [r for r in history if r['method'] == c['method'] and r['seed'] == c['seed']]
        check(sorted(r['epoch'] for r in h) == list(range(1, 31)), 'alpha=.5 历史不完整。')
        best = max(sorted(h, key=lambda r:r['epoch']), key=lambda r:(r['validation']['macro_f1'], -r['validation']['nll']))
        check(best['epoch'] == c['epoch'] and abs(best['validation']['macro_f1']-c['validation_macro_f1']) < 1e-12, '基线选择记录和历史不一致。')
        baseline.append(dict(method=c['method'], seed=c['seed'], alpha=.5, best_epoch=c['epoch'],
                             validation_macro_f1=c['validation_macro_f1'], validation_nll=best['validation']['nll'],
                             checkpoint=str((work / 'students' / c['checkpoint']).resolve()), checkpoint_sha256=c['checkpoint_sha256'],
                             status='reused_complete', origin='v1 alpha=0.5'))
    data = {}
    for split in ['train', 'val_select']:
        path = work / f'features/{split}.npz'
        meta = read_json(work / f'features/{split}_meta.json')
        check(digest(path) == meta['npz_sha256'] == lock['feature_hashes'][split] == selection['feature_hashes'][split], '特征哈希不匹配。')
        check(meta['manifest_sha256'] == digest(work / f'handoff/manifests/{split}.jsonl'), '清单哈希不匹配。')
        check(meta['revision'] == read_json(work / 'features/clip_revision.json')['revision'], 'CLIP revision 不一致。')
        rows = read_rows(work / f'handoff/manifests/{split}.jsonl')
        with np.load(path, allow_pickle=False) as a:
            check(a['ids'].tolist() == [r['sample_id'] for r in rows], '特征 sample_id 不一致。')
            check(a['y'].tolist() == [r['label'] for r in rows], '特征标签不一致。')
            check(a['image_ids'].astype(str).tolist() == [str(r['image_id']) for r in rows], '特征图片 ID 不一致。')
            check(np.isfinite(a['x']).all() and np.isin(a['y'], [0,1,2]).all(), '特征或标签无效。')
            data[split] = dict(x=a['x'].copy(), y=a['y'].copy(), rows=rows)
    teacher = read_rows(work / 'handoff/predictions/train/jev.jsonl')
    by_id = {r['sample_id']: r for r in teacher}
    check(len(by_id) == len(teacher) and set(by_id) == {r['sample_id'] for r in data['train']['rows']}, 'Teacher 训练样本不一致。')
    aligned = [by_id[r['sample_id']] for r in data['train']['rows']]
    for t, r in zip(aligned, data['train']['rows']):
        check(all(t[k] == r[k] for k in ['label', 'image_id', 'hypothesis', 'split']), 'Teacher 内容不一致。')
    z = np.asarray([r['logits'] for r in aligned], dtype=float)
    check(z.shape == (len(aligned), 3) and np.isfinite(z).all(), 'Teacher logits 无效。')
    check(digest(work / 'handoff/predictions/train/jev.jsonl') == lock['teacher_train_sha256'] == selection['teacher_train_sha256'], 'Teacher 来源不一致。')
    check(digest(work / 'handoff/results/calibration.json') == lock['calibration_sha256'] == selection['calibration_sha256'], '校准来源不一致。')
    params = read_json(work / 'handoff/results/calibration.json')['models']['jev']['selected_params']
    calibrated = (z + np.array([params['b'], 0, 0])) / params['T']
    inputs = {name:digest(work/name) for name in required}
    protocol = dict(methods=METHODS, seeds=SEEDS, alphas=ALPHAS, new_alphas=[.25,.75], tau=2.,
                    alpha_definition='KD weight: (1-alpha)*CE + alpha*tau^2*KL(teacher||student)',
                    fixed_settings=expected_settings, input_sha256=inputs,
                    selection='val_select Macro-F1, tie lower NLL, then earliest epoch',
                    candidates='only after all 18 complete; per method highest mean F1, tie lower mean NLL then closest to .5 then smaller alpha',
                    no_test_evaluation=True)
    protocol_path = experiment / 'protocol.json'
    if protocol_path.exists():
        check(read_json(protocol_path) == protocol, '实验输入或协议已改变，请使用新实验目录。')
    else:
        atomic_json(protocol_path, protocol)
    atomic_json(experiment / 'baseline_reused.json', baseline)
    return dict(work=work, experiment=experiment, data=data, targets={'KD_raw':z, 'KD_calibrated':calibrated},
                baseline=baseline, original_lock=lock, protocol_sha256=digest(protocol_path))


def report(context):
    """Every call publishes a new report snapshot; incomplete comparisons never nominate alpha."""
    experiment = context['experiment']
    rows = list(context['baseline'])
    progress = []
    for method in METHODS:
        for alpha in [.25, .75]:
            for seed in SEEDS:
                name = run_name(method, alpha, seed)
                folder = experiment / 'runs' / name
                done, resume = folder / 'done.json', folder / 'resume.pt'
                if done.exists():
                    result = read_json(done)
                    check(result['protocol_sha256'] == context['protocol_sha256'], '完成项协议不一致。')
                    check((result['method'],result['alpha'],result['seed']) == (method,alpha,seed), '完成项标识不一致。')
                    checkpoint = folder / 'best.pt'
                    check(digest(checkpoint) == result['checkpoint_sha256'], '完成项 checkpoint 已改变。')
                    check(digest(folder/'history.json') == result['history_sha256'], '完成项历史已改变。')
                    history = read_json(folder/'history.json')
                    check(sorted(r['epoch'] for r in history) == list(range(1,31)), '完成项没有完整30轮历史。')
                    check(all((r['method'],r['alpha'],r['seed']) == (method,alpha,seed) for r in history), '完成项历史标识不一致。')
                    best = max(sorted(history,key=lambda r:r['epoch']), key=lambda r:(r['validation']['macro_f1'],-r['validation']['nll']))
                    check(best['epoch'] == result['best_epoch'] and best['validation']['macro_f1'] == result['validation_macro_f1']
                          and best['validation']['nll'] == result['validation_nll'], '完成项最佳epoch或指标不一致。')
                    rows.append(result)
                    status, epoch = 'complete', 30
                elif resume.exists():
                    # Written immediately after the atomic checkpoint. A mismatch is shown honestly.
                    info = read_json(folder / 'progress.json') if (folder / 'progress.json').exists() else {}
                    valid = info.get('resume_sha256') == digest(resume)
                    status = 'resumable' if valid else 'resumable_needs_state_read'
                    epoch = info.get('saved_epoch') if valid else None
                else:
                    status, epoch = 'pending', 0
                progress.append(dict(method=method, alpha=alpha, seed=seed, status=status,
                                     saved_epoch=epoch, next_epoch=(epoch+1 if epoch is not None and epoch < 30 else None),
                                     recovery_file=str(resume) if resume.exists() else '',
                                     note='仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。'))
    detail = pd.DataFrame(rows)
    check(not detail.duplicated(['method','alpha','seed']).any(), '重复完成项。')
    summaries = []
    for method in METHODS:
        for alpha in ALPHAS:
            group = detail[(detail.method == method) & (detail.alpha == alpha)]
            complete = len(group) == 3 and set(group.seed) == set(SEEDS)
            summaries.append(dict(method=method, alpha=alpha, completed_seeds=len(group), expected_seeds=3,
                                 status='complete' if complete else 'incomplete',
                                 mean_macro_f1_pct=100*group.validation_macro_f1.mean() if complete else None,
                                 std_macro_f1_pct=100*group.validation_macro_f1.std(ddof=1) if complete else None,
                                 mean_nll=group.validation_nll.mean() if complete else None,
                                 mean_plus_minus_std=(f'{100*group.validation_macro_f1.mean():.3f} ± {100*group.validation_macro_f1.std(ddof=1):.3f}') if complete else '未完成，不汇总'))
    summary = pd.DataFrame(summaries)
    all_complete = len(detail) == 18 and summary.status.eq('complete').all()
    candidates = dict(status='complete' if all_complete else 'incomplete', candidates=[],
                      reason='全部18组完成后才按两种KD分别选择下一轮候选；不是测试集结论。')
    paired = []
    if all_complete:
        for method in METHODS:
            group = summary[summary.method == method].copy()
            group['distance_to_half'] = (group.alpha-.5).abs()
            best = group.sort_values(['mean_macro_f1_pct','mean_nll','distance_to_half','alpha'], ascending=[False,True,True,True]).iloc[0]
            candidates['candidates'].append(dict(method=method, alpha=float(best.alpha),
                mean_macro_f1_pct=float(best.mean_macro_f1_pct), std_macro_f1_pct=float(best.std_macro_f1_pct),
                interpretation='验证集候选，不声称统计显著或已确认泛化提升'))
            baseline = detail[(detail.method == method) & (detail.alpha == .5)].set_index('seed')
            for alpha in [.25, .75]:
                g = detail[(detail.method == method) & (detail.alpha == alpha)].set_index('seed')
                for seed in SEEDS:
                    paired.append(dict(method=method, alpha=alpha, seed=seed,
                                       macro_f1_delta_pp=100*(g.loc[seed,'validation_macro_f1']-baseline.loc[seed,'validation_macro_f1'])))
    stamp = datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    dest = experiment / 'reports' / stamp
    dest.mkdir(parents=True, exist_ok=False)
    detail.to_csv(dest/'best_epochs_and_scores.csv', index=False, encoding='utf-8-sig')
    summary.to_csv(dest/'validation_mean_std.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(progress).to_csv(dest/'completion_and_resume.csv', index=False, encoding='utf-8-sig')
    if paired:
        pd.DataFrame(paired).to_csv(dest/'paired_deltas_vs_alpha05.csv', index=False, encoding='utf-8-sig')
    atomic_json(dest/'candidate_alpha.json', candidates)
    atomic_json(dest/'snapshot.json', dict(all_complete=bool(all_complete), new_completed=len(detail)-6,
                new_total=12, reused=6, protocol_sha256=context['protocol_sha256']))
    print(f'新训练完成 {len(detail)-6}/12；复用 6/6。报告：{dest}')
    if not all_complete:
        print('尚未全部完成：不宣布最优，不给出候选 alpha。')
    return summary, detail, pd.DataFrame(progress), candidates, dest


def train_sweep(context, max_new_runs=None, max_minutes=None):
    import torch
    import torch.nn.functional as F
    from packaging.version import Version
    check(Version(torch.__version__.split('+')[0]) >= Version('2.6'), '需要 PyTorch >= 2.6。')
    check(max_new_runs is None or isinstance(max_new_runs, int) and max_new_runs >= 0, 'max_new_runs 必须为非负整数或 None。')
    check(max_minutes is None or max_minutes > 0, 'max_minutes 必须为正数或 None。')
    original = context['original_lock']
    # Reuse across environments is blocked rather than silently confounding alpha with software changes.
    check(torch.__version__ == original['software']['torch'] and np.__version__ == original['software']['numpy']
          and torch.version.cuda == original['torch_cuda'],
          f"请使用原训练环境后继续：torch={original['software']['torch']}, numpy={original['software']['numpy']}, CUDA={original['torch_cuda']}。不混用环境比较 alpha。")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    check(device == 'cuda', '原训练使用 CUDA；请切换到原训练 GPU 环境。')
    experiment = context['experiment']
    implementation = experiment / 'alpha_sweep_runtime.py'
    check(implementation.is_file(), '缺少本轮运行代码快照，请先运行 notebook 的代码快照单元格。')
    runtime = dict(protocol_sha256=context['protocol_sha256'], code_sha256=digest(implementation),
                   torch=str(torch.__version__), numpy=str(np.__version__), cuda=torch.version.cuda,
                   device=device, gpu=torch.cuda.get_device_name(), gpu_count=torch.cuda.device_count(),
                   cudnn=torch.backends.cudnn.version())
    runtime_path = experiment/'runtime_lock.json'
    if runtime_path.exists():
        check(read_json(runtime_path) == runtime, '恢复环境或代码变化，请恢复原环境；不要混合结果。')
    else:
        atomic_json(runtime_path, runtime)
    # Concurrent runs against this directory must never write the same checkpoint.
    owner = experiment/'RUNNING.lock'
    with owner.open('x', encoding='utf-8') as f:
        f.write(json.dumps(dict(pid=os.getpid(), note='只在确认旧会话已经停止后手动删除此锁。')))
    def atomic_torch(path, value):
        temp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
        with temp.open('xb') as f:
            torch.save(value, f); f.flush(); os.fsync(f.fileno())
        temp.replace(path)
    def head(dimension):
        return torch.nn.Sequential(torch.nn.Linear(dimension,512),torch.nn.ReLU(),
                                   torch.nn.Dropout(.1),torch.nn.Linear(512,3))
    started = time.monotonic()
    finished = 0
    def deadline():
        return max_minutes is not None and time.monotonic()-started >= 60*max_minutes
    try:
        a, b = context['data']['train'], context['data']['val_select']
        x = torch.tensor(a['x'], device=device)
        y = torch.tensor(a['y'], dtype=torch.long, device=device)
        xv = torch.tensor(b['x'], device=device)
        # Equal initialization within each seed; alpha never participates in seeding.
        for seed in SEEDS:
            for method in METHODS:
                for alpha in [.25,.75]:
                    name = run_name(method,alpha,seed)
                    folder = experiment/'runs'/name
                    folder.mkdir(parents=True, exist_ok=True)
                    done, resume = folder/'done.json', folder/'resume.pt'
                    if done.exists():
                        result = read_json(done)
                        check(result['protocol_sha256'] == context['protocol_sha256'] and digest(folder/'best.pt') == result['checkpoint_sha256'], '完成项校验失败。')
                        continue
                    if (max_new_runs is not None and finished >= max_new_runs) or deadline():
                        return
                    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
                    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
                    model = head(x.shape[1]).to(device)
                    optimizer = torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)
                    qt = torch.tensor(softmax(context['targets'][method]/2.),dtype=torch.float32,device=device)
                    config = dict(runtime=runtime,method=method,alpha=alpha,seed=seed)
                    history=[]; best=(-1.,-float('inf')); best_epoch=None; best_state=None; start=1
                    if resume.exists():
                        state = torch.load(resume,map_location='cpu',weights_only=True)
                        check(state['config'] == config, '恢复点配置不一致。')
                        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
                        best=tuple(state['best']); best_epoch=state['best_epoch']; best_state=state['best_state']
                        history=state['history']; start=state['epoch']+1
                        random.setstate(state['random'])
                        rng=state['numpy']; np.random.set_state((rng[0],np.array(rng[1],dtype=np.uint32),rng[2],rng[3],rng[4]))
                        torch.set_rng_state(state['torch_rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
                        print(f'{name} 从 epoch {start} 恢复。', flush=True)
                    for epoch in range(start,31):
                        model.train(); total=0.
                        for idx in torch.randperm(len(y),device=device).split(128):
                            logits=model(x[idx]); ce=F.cross_entropy(logits,y[idx])
                            loss=(1-alpha)*ce+alpha*4.*F.kl_div(F.log_softmax(logits/2.,dim=-1),qt[idx],reduction='batchmean')
                            optimizer.zero_grad(); loss.backward(); optimizer.step()
                            total+=loss.item()*len(idx)
                        model.eval()
                        with torch.inference_mode():
                            p=model(xv).softmax(-1).cpu().numpy()
                        check(np.isfinite(p).all() and np.isfinite(total), '训练产生非有限数值，停止并保留恢复点。')
                        metric=score(b['y'],p); key=(metric['macro_f1'],-metric['nll'])
                        history.append(dict(method=method,alpha=alpha,seed=seed,epoch=epoch,loss=total/len(y),validation=metric))
                        if key>best:
                            best=key; best_epoch=epoch
                            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                        print(f'{name} epoch {epoch}/30: val Macro-F1={metric["macro_f1"]:.6f}',flush=True)
                        if epoch%5==0:
                            rng=np.random.get_state()
                            atomic_torch(resume,dict(config=config,epoch=epoch,model=model.state_dict(),optimizer=optimizer.state_dict(),
                                best=best,best_epoch=best_epoch,best_state=best_state,history=history,
                                random=random.getstate(),numpy=(rng[0],rng[1].tolist(),rng[2],rng[3],rng[4]),
                                torch_rng=torch.get_rng_state(),cuda_rng=torch.cuda.get_rng_state_all()))
                            atomic_json(folder/'progress.json',dict(saved_epoch=epoch,next_epoch=epoch+1,
                                        resume_sha256=digest(resume),best_epoch_so_far=best_epoch))
                            # Respect time budget only at the original 5-epoch checkpoint boundary.
                            if deadline() and epoch<30:
                                return
                    check(len(history)==30 and best_epoch is not None, '训练记录不完整。')
                    atomic_torch(folder/'best.pt',best_state)
                    atomic_json(folder/'history.json',history)
                    result=dict(method=method,alpha=alpha,seed=seed,best_epoch=best_epoch,
                                validation_macro_f1=best[0],validation_nll=-best[1],checkpoint=str((folder/'best.pt').resolve()),
                                checkpoint_sha256=digest(folder/'best.pt'),history_sha256=digest(folder/'history.json'),
                                protocol_sha256=context['protocol_sha256'],runtime_lock_sha256=digest(runtime_path),
                                status='complete',origin='new alpha sweep')
                    atomic_json(done,result)
                    finished+=1
    except KeyboardInterrupt:
        print('已中断；保留最近每5个epoch的完整恢复点，下次使用同一 EXPERIMENT_DIR。')
    finally:
        owner.unlink(missing_ok=True)
        report(context)
'''
exec(compile(RUNTIME_SOURCE, 'alpha_sweep_runtime.py', 'exec'))


## 3．核验并登记六组复用结果

核对特征、训练 Teacher、校准、基线 checkpoint、30 轮历史与最佳 epoch。第一次建立协议锁，后续恢复必须与该锁一致。不读取测试报告，也不重新生成 CLIP 特征。


In [4]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

matches = list(
    Path('/content/drive/MyDrive').rglob('CLIP_Jev_Student_v1')
)
for p in matches:
    print(p)

if not matches:
    print("Drive 中没找到：请先把 student_delivery.zip 上传到“我的云端硬盘”。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/CLIP_Jev_Student_v1


In [5]:
context = prepare(SOURCE, EXPERIMENT_DIR)
code_path = EXPERIMENT_DIR / 'alpha_sweep_runtime.py'
if code_path.exists():
    check(code_path.read_text(encoding='utf-8') == RUNTIME_SOURCE, '运行代码已经变化；恢复原 notebook 或使用新实验目录。')
else:
    with code_path.open('x', encoding='utf-8', newline='') as f:
        f.write(RUNTIME_SOURCE)
print('复用 alpha=0.5：')
from IPython.display import display
display(pd.DataFrame(context['baseline'])[['method','alpha','seed','best_epoch','validation_macro_f1','validation_nll']])
summary, detail, progress, candidates, report_dir = report(context)
display(summary)
display(progress)


复用 alpha=0.5：


,method,alpha,seed,best_epoch,validation_macro_f1,validation_nll
0,KD_raw,0.5,20261010,22,0.658821,0.893938
1,KD_calibrated,0.5,20261010,24,0.657853,0.875294
2,KD_raw,0.5,20261011,10,0.662045,0.863223
3,KD_calibrated,0.5,20261011,17,0.661169,0.846074
4,KD_raw,0.5,20261012,19,0.656630,0.886356
5,KD_calibrated,0.5,20261012,19,0.651960,0.871387


新训练完成 0/12；复用 6/6。报告：/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1/reports/20261012_171124_dd672c0a
尚未全部完成：不宣布最优，不给出候选 alpha。


,method,alpha,completed_seeds,expected_seeds,status,mean_macro_f1_pct,std_macro_f1_pct,mean_nll,mean_plus_minus_std
0,KD_raw,0.25,0,3,incomplete,NaN,NaN,NaN,未完成，不汇总
1,KD_raw,0.50,3,3,complete,65.916551,0.272395,0.881172,65.917 ± 0.272
2,KD_raw,0.75,0,3,incomplete,NaN,NaN,NaN,未完成，不汇总
3,KD_calibrated,0.25,0,3,incomplete,NaN,NaN,NaN,未完成，不汇总
4,KD_calibrated,0.50,3,3,complete,65.699377,0.466421,0.864252,65.699 ± 0.466
5,KD_calibrated,0.75,0,3,incomplete,NaN,NaN,NaN,未完成，不汇总


,method,alpha,seed,status,saved_epoch,next_epoch,recovery_file,note
0,KD_raw,0.25,20261010,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
1,KD_raw,0.25,20261011,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
2,KD_raw,0.25,20261012,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
3,KD_raw,0.75,20261010,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
4,KD_raw,0.75,20261011,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
5,KD_raw,0.75,20261012,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
6,KD_calibrated,0.25,20261010,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
7,KD_calibrated,0.25,20261011,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
8,KD_calibrated,0.25,20261012,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
9,KD_calibrated,0.75,20261010,pending,0,1,,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。


## 4．运行或恢复新增的十二次训练

默认最多运行 180 分钟；在预算到达后的下一个 5-epoch 保存点退出，可能略超出时间预算。`MAX_NEW_RUNS` 可限制本次完成的训练次数（例如 4），`None` 表示不限；两项预算只控制何时暂停，不改变每次训练的 30-epoch 预算。

中断或当天未跑完：再次打开本 notebook，使用同一 `EXPERIMENT_DIR`，从第 1 节顺序运行。已完成项校验后跳过，未完成项从最近的完整保存点继续。中断于一个 epoch 或保存点之间时，未保存的部分会重跑。

如果会话被强制终止，`RUNNING.lock` 可能残留：先确认旧会话确已停止，再手动删除新实验目录中的该锁文件。不要删除 `runs/`、`resume.pt` 或协议文件，不要让两个会话同时写同一实验目录。


In [6]:
import torch
import numpy as np

print("当前 PyTorch：", torch.__version__)
print("当前 NumPy：", np.__version__)
print("当前 PyTorch CUDA：", torch.version.cuda)
print("GPU 可用：", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))

print("\n原训练环境：")
print("PyTorch：2.11.0+cu130")
print("NumPy：2.2.6")
print("PyTorch CUDA：13.0")

当前 PyTorch： 2.11.0+cu130
当前 NumPy： 2.2.6
当前 PyTorch CUDA： 13.0
GPU 可用： True
GPU： Tesla T4

原训练环境：
PyTorch：2.11.0+cu130
NumPy：2.2.6
PyTorch CUDA：13.0


In [7]:
RUN_TRAINING = True
MAX_NEW_RUNS = None  # 例如 4；0 表示不训练
MAX_MINUTES = 180   # None 表示不设时间预算

if RUN_TRAINING:
    train_sweep(context, max_new_runs=MAX_NEW_RUNS, max_minutes=MAX_MINUTES)
else:
    print('本次仅查看已有结果，不启动训练。')


KD_raw_alpha0.25_20261010 epoch 1/30: val Macro-F1=0.470717
KD_raw_alpha0.25_20261010 epoch 2/30: val Macro-F1=0.577265
KD_raw_alpha0.25_20261010 epoch 3/30: val Macro-F1=0.591757
KD_raw_alpha0.25_20261010 epoch 4/30: val Macro-F1=0.614181
KD_raw_alpha0.25_20261010 epoch 5/30: val Macro-F1=0.610673
KD_raw_alpha0.25_20261010 epoch 6/30: val Macro-F1=0.607597
KD_raw_alpha0.25_20261010 epoch 7/30: val Macro-F1=0.637766
KD_raw_alpha0.25_20261010 epoch 8/30: val Macro-F1=0.600885
KD_raw_alpha0.25_20261010 epoch 9/30: val Macro-F1=0.628768
KD_raw_alpha0.25_20261010 epoch 10/30: val Macro-F1=0.632545
KD_raw_alpha0.25_20261010 epoch 11/30: val Macro-F1=0.641209
KD_raw_alpha0.25_20261010 epoch 12/30: val Macro-F1=0.630912
KD_raw_alpha0.25_20261010 epoch 13/30: val Macro-F1=0.633594
KD_raw_alpha0.25_20261010 epoch 14/30: val Macro-F1=0.637806
KD_raw_alpha0.25_20261010 epoch 15/30: val Macro-F1=0.616038
KD_raw_alpha0.25_20261010 epoch 16/30: val Macro-F1=0.631833
KD_raw_alpha0.25_20261010 epoch 1

## 5．交付均值±标准差、最佳 epoch 与恢复清单

任意时刻可运行此格。均值±标准差的指标单位为百分数，标准差采用三个种子的样本标准差（ddof=1）。只有三种子均完成的设置才显示均值；候选必须等所有设置完成才生成。

候选选择预先固定为：**两种 KD 各选一个 alpha**，按验证集平均 Macro-F1 从高到低；严格平局依次选平均 NLL 较低、较接近 0.5、alpha 较小者。不会根据部分结果更改规则，候选可以仍是 0.5。附与 0.5 同种子的差值，便于检查提升是否由单一种子主导；不把均值排序解释为统计显著。


In [8]:
summary, detail, progress, candidates, report_dir = report(context)
print('验证集汇总：')
display(summary[['method','alpha','completed_seeds','status','mean_plus_minus_std']])
print('已完成/复用设置的最佳 epoch：')
display(detail[['method','alpha','seed','best_epoch','validation_macro_f1','validation_nll','origin']].sort_values(['method','alpha','seed']))
print('新增训练完成项和恢复点：')
display(progress)
print('下一轮候选：')
if candidates['status'] == 'complete':
    display(pd.DataFrame(candidates['candidates']))
else:
    print('待全部18组完成后确定；当前不宣布最优。')
print('本次报告快照：', report_dir)


新训练完成 12/12；复用 6/6。报告：/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1/reports/20261012_171322_9d137e91
验证集汇总：


,method,alpha,completed_seeds,status,mean_plus_minus_std
0,KD_raw,0.25,3,complete,64.602 ± 0.804
1,KD_raw,0.50,3,complete,65.917 ± 0.272
2,KD_raw,0.75,3,complete,66.944 ± 0.267
3,KD_calibrated,0.25,3,complete,64.387 ± 0.203
4,KD_calibrated,0.50,3,complete,65.699 ± 0.466
5,KD_calibrated,0.75,3,complete,66.924 ± 0.596


已完成/复用设置的最佳 epoch：


,method,alpha,seed,best_epoch,validation_macro_f1,validation_nll,origin
12,KD_calibrated,0.25,20261010,11,0.644103,0.869068,new alpha sweep
13,KD_calibrated,0.25,20261011,8,0.645778,0.857211,new alpha sweep
14,KD_calibrated,0.25,20261012,10,0.641732,0.865795,new alpha sweep
1,KD_calibrated,0.50,20261010,24,0.657853,0.875294,v1 alpha=0.5
3,KD_calibrated,0.50,20261011,17,0.661169,0.846074,v1 alpha=0.5
5,KD_calibrated,0.50,20261012,19,0.651960,0.871387,v1 alpha=0.5
15,KD_calibrated,0.75,20261010,12,0.668992,0.838839,new alpha sweep
16,KD_calibrated,0.75,20261011,10,0.675318,0.832046,new alpha sweep
17,KD_calibrated,0.75,20261012,30,0.663402,0.863576,new alpha sweep
6,KD_raw,0.25,20261010,11,0.641209,0.882512,new alpha sweep


新增训练完成项和恢复点：


,method,alpha,seed,status,saved_epoch,next_epoch,recovery_file,note
0,KD_raw,0.25,20261010,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
1,KD_raw,0.25,20261011,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
2,KD_raw,0.25,20261012,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
3,KD_raw,0.75,20261010,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
4,KD_raw,0.75,20261011,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
5,KD_raw,0.75,20261012,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
6,KD_calibrated,0.25,20261010,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
7,KD_calibrated,0.25,20261011,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
8,KD_calibrated,0.25,20261012,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。
9,KD_calibrated,0.75,20261010,complete,30,None,/content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1...,仅以上次成功保存的完整 epoch 为恢复点；未保存部分会重跑。


下一轮候选：


,method,alpha,mean_macro_f1_pct,std_macro_f1_pct,interpretation
0,KD_raw,0.75,66.943917,0.266675,验证集候选，不声称统计显著或已确认泛化提升
1,KD_calibrated,0.75,66.923744,0.596167,验证集候选，不声称统计显著或已确认泛化提升


本次报告快照： /content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1/reports/20261012_171322_9d137e91


## 6．打包当前报告（未完成也可交付进度）

报告 ZIP 包含本次汇总、最佳 epoch、完成与恢复清单、候选状态、协议和代码快照。完整模型和恢复状态保留在新实验目录 `runs/` 下，不重复压缩；如需迁移训练，请保留整个实验目录以及原始 SOURCE。每次报告有独立时间标识，不覆盖先前报告。

未完成时交付 `completion_and_resume.csv`、`snapshot.json` 和候选状态 `incomplete`，不要将此时的表格描述为最终比较结果。本 notebook 没有测试集评估入口。


In [9]:
bundle = report_dir / ('alpha_validation_complete.zip' if candidates['status']=='complete' else 'alpha_validation_progress.zip')
files = list(report_dir.glob('*.csv')) + list(report_dir.glob('*.json'))
if not bundle.exists():
    with zipfile.ZipFile(bundle, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.name)
        for name in ['protocol.json','baseline_reused.json','alpha_sweep_runtime.py']:
            z.write(EXPERIMENT_DIR/name, name)
        if (EXPERIMENT_DIR/'runtime_lock.json').exists():
            z.write(EXPERIMENT_DIR/'runtime_lock.json', 'runtime_lock.json')
print('可交付报告包：', bundle)
print('训练模型与恢复状态：', EXPERIMENT_DIR/'runs')


可交付报告包： /content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1/reports/20261012_171322_9d137e91/alpha_validation_complete.zip
训练模型与恢复状态： /content/drive/MyDrive/CLIP_Jev_alpha_sweep_v1/runs
